# C1 AML False-Positive Reducer — End-to-End Demo

1. Generate 10,000 synthetic AML alerts (~92% FP / 8% TP).
2. Run the naive baseline (keep everything).
3. Run the LangGraph + Ollama (qwen2.5:7b) triage agent on a 200-alert sample.
4. Compare metrics side-by-side and visualise the volume / recall trade-off.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from c1_aml_fp_reducer import baseline, eval as eval_mod, synth, triage

## 1. Synthetic alert generation

In [ ]:
df = synth.generate_alerts(n=10_000, seed=42)
print(synth.summary(df))
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df.groupby('alert_reason')['is_true_positive'].mean().sort_values().plot.barh(ax=ax[0])
ax[0].set_title('TP rate by alert reason'); ax[0].set_xlabel('P(true positive)')
df['customer_segment'].value_counts().plot.pie(ax=ax[1], autopct='%.0f%%')
ax[1].set_title('Customer segment mix'); ax[1].set_ylabel('')
plt.tight_layout(); plt.show()

## 2. Naive baseline

In [ ]:
base_metrics = eval_mod.evaluate(df, baseline.baseline_label_kept(df), name='baseline')
base_metrics.to_dict()

## 3. LangGraph + Ollama triage on 200-alert sample

Requires `ollama serve` running locally and `qwen2.5:7b` pulled. If unavailable, the agent falls back to a deterministic rule-based scorer (with a warning).

In [ ]:
sample = df.head(200).copy()
triaged = triage.triage_dataframe(sample, progress=True)
triaged[['alert_id','alert_reason','fp_confidence','decision','is_true_positive']].head(15)

In [ ]:
tri_metrics = eval_mod.evaluate(sample, triage.triage_label_kept(triaged), name='llm_triage')
print(eval_mod.format_report([base_metrics, tri_metrics]))

## 4. Volume / recall trade-off chart

In [ ]:
labels = ['baseline', 'llm_triage']
vol = [base_metrics.volume_reduction_pct, tri_metrics.volume_reduction_pct]
rec = [base_metrics.recall * 100, tri_metrics.recall * 100]

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar([x for x in range(len(labels))], vol, color='steelblue', alpha=0.7, label='Volume reduction %')
ax1.set_xticks(range(len(labels))); ax1.set_xticklabels(labels)
ax1.set_ylabel('Volume reduction %', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(labels, rec, 'o-', color='crimson', label='Recall %', linewidth=2)
ax2.set_ylabel('Recall %', color='crimson'); ax2.set_ylim(80, 101)
plt.title('Volume reduction vs. true-positive recall')
plt.tight_layout(); plt.show()

**Bottom line.** The triage agent dismisses a large fraction of obviously-benign alerts at near-perfect recall, freeing analyst hours for the genuinely interesting cases. See `docs/whitepaper.md` for the deeper writeup and `data/metrics.json` for live numbers.